## Step 1) Install libraries

In [1]:
!pip install -q llama-index-core
!pip install -q llama-index-llms-groq
!pip install -q llama-index-readers-file
!pip install -q llama-index-embeddings-huggingface
!pip install -q pypdf

## Step 2) Load API keys

## Step 3) Set up the LLM

In [15]:
# Import Groq LLM from LlamaIndex
from llama_index.llms.groq import Groq

# Choose the LLM model
model = "llama-3.3-70b-versatile"

# Create the LLM object
llm = Groq(model=model)

## Step 4) Set up the embedding model

In [16]:
# Import Hugging Face embedding model
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Choose an embedding model
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"

# Create embedding model object
embeddings = HuggingFaceEmbedding(
    model_name=embedding_model_name,
    cache_folder="/content/embedding_model"
)

2026-03-26 00:27:52,773 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


## Step 5) Create the text splitter

In [17]:
# Import sentence splitter
from llama_index.core.node_parser import SentenceSplitter

# Split text into chunks with overlap
text_splitter = SentenceSplitter(
    chunk_size=800,
    chunk_overlap=150
)

## Step 6) Build the prompt

In [18]:
# Create memory  


# Import prompt template
from llama_index.core.prompts import PromptTemplate

# Create a simple RAG prompt
input_template = """
Here is the context:
{context_str}

Answer the question based only on the context above.
If the answer is not in the context, say: "The answer is not found in the document."

Question: {query_str}
Answer:
"""

# Create prompt object
prompt = PromptTemplate(template=input_template)

## Step 7) Load your PDF documents

In [19]:
# Import file reader
from llama_index.core import SimpleDirectoryReader

# Load only PDF files from the data folder
documents = SimpleDirectoryReader(
    "./data",
    required_exts=[".pdf"]
).load_data(show_progress=True)

# Check how many documents were loaded
print(f"Loaded {len(documents)} document(s).")

Loading files: 100%|██████████| 1/1 [00:03<00:00,  3.40s/it]

Loaded 36 document(s).


## Step 8) Create the vector index

In [20]:
# Import vector index
from llama_index.core import VectorStoreIndex

# Build the vector index from documents
vector_index = VectorStoreIndex.from_documents(
    documents,
    transformations=[text_splitter],
    embed_model=embeddings
)

In [21]:
# Import chat memory
from llama_index.core.memory import ChatMemoryBuffer
memory = ChatMemoryBuffer.from_defaults(token_limit=1500)

# Create chat engine with memory
chat_engine = vector_index.as_chat_engine(
    chat_mode="context",
    llm=llm,
    memory=memory,
    similarity_top_k=2)

## Step 9) Create the query engine

In [22]:
# Create the query engine
query_engine = vector_index.as_query_engine(
    llm=llm,
    text_qa_template=prompt,
    similarity_top_k=2,
    response_mode="tree_summarize"
)

## Step 10) Ask a question

In [23]:
# Define a question
question = "What is the green hause gas?"

# Query the RAG system
answer = query_engine.query(question)

# Print the final answer
print(answer)

2026-03-26 00:28:16,426 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


The greenhouse gases mentioned are carbon dioxide (CO2), methane, and nitrous oxide. These gases have increased significantly in the atmosphere since the Industrial Revolution, with CO2 playing the largest role in warming the Earth.


## Step 11) See only the response text

In [24]:
# Print only the answer text
print(answer.response)

The greenhouse gases mentioned are carbon dioxide (CO2), methane, and nitrous oxide. These gases have increased significantly in the atmosphere since the Industrial Revolution, with CO2 playing the largest role in warming the Earth.


## Step 12) See which text chunks were used

In [25]:
# Show the source nodes used for the answer
print(answer.source_nodes)

[NodeWithScore(node=TextNode(id_='9c7e1658-42d6-4190-b044-ce27506c70e1', embedding=None, metadata={'page_label': '18', 'file_name': 'climate-change-evidence-causes.pdf', 'file_path': 'c:\\Bootcamp\\AI\\data\\climate-change-evidence-causes.pdf', 'file_type': 'application/pdf', 'file_size': 3679443, 'creation_date': '2026-03-25', 'last_modified_date': '2026-03-25'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='929aa5eb-3722-4e80-8abe-8da9510082f3', node_type='4', metadata={'page_label': '18', 'file_name': 'climate-change-evidence-causes.pdf', 'file_path': 'c:\\Bootcamp\\AI\\data\\climate-change-evidence-causes.pdf', 'file_type': 'application/pdf', 'file_size': 3679443, 'creation_date': '2026-03-25', 'last_

In [26]:
# Simple chatbot loop

while True:
    question = input("Ask your question (type 'end' to exit): ").strip()

    if question.lower() == "end":
        print("Goodbye")
        break

    if not question:
        print("Please enter a valid question.")
        print("-" * 50)
        continue

    try:
        response = chat_engine.chat(question)

        print("\nQuestion:")
        print(question)
        print("\nAnswer:")
        print(response.response)
        print("-" * 50)

    except Exception as e:
        print("Error:", e)
        print("Try another question.")
        print("-" * 50)

Please enter a valid question.
--------------------------------------------------


2026-03-26 00:28:50,406 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



Question:
How do greenhouse gases affect the Earth's temperature?

Answer:
According to the text, greenhouse gases, including water vapor, carbon dioxide, methane, and nitrous oxide, absorb and emit heat energy in all directions, including downwards, keeping the Earth's surface and lower atmosphere warm. This is known as the greenhouse effect. Without this effect, the Earth's average surface temperature would be tens of degrees colder than it is today.

The greenhouse gases absorb some of the infrared energy that the Earth emits, and re-emit it in all directions, trapping heat and warming the surface. As the concentration of greenhouse gases in the atmosphere increases, the atmosphere becomes more effective at preventing heat from escaping into space, leading to an increase in the Earth's surface temperature.

In particular, carbon dioxide (CO2) has a strong heat-trapping band centered at a wavelength of 15 micrometers, and as CO2 concentrations increase, more energy is absorbed in th

2026-03-26 00:29:06,001 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



Question:
How do greenhouse gases affect the Earth's temperature?

Answer:
Greenhouse gases, such as carbon dioxide (CO2), water vapor, methane, and nitrous oxide, affect the Earth's temperature by trapping heat from the sun. Here's a step-by-step explanation:

1. **The Sun's energy enters the Earth's atmosphere**: The Sun emits solar radiation, which enters the Earth's atmosphere.
2. **Some energy is reflected back into space**: A portion of the solar radiation is reflected back into space by the Earth's surface, atmosphere, and clouds.
3. **The Earth's surface absorbs energy**: The remaining energy is absorbed by the Earth's surface, warming it up.
4. **The Earth's surface emits infrared radiation**: As the Earth's surface warms, it emits infrared radiation (heat) back into the atmosphere.
5. **Greenhouse gases absorb and re-emit heat**: Greenhouse gases in the atmosphere, such as CO2, absorb some of the infrared radiation emitted by the Earth's surface.
6. **Heat is trapped**: The 

...


Question:
what is the first question?

Answer:
Your first question was: "Why is carbon dioxide important in climate change?"
--------------------------------------------------


2026-03-26 00:31:57,790 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



Question:
How do greenhouse gases affect the Earth's temperature?

Answer:
Greenhouse gases, such as carbon dioxide (CO2), methane (CH4), and water vapor (H2O), affect the Earth's temperature by trapping heat from the sun and preventing it from escaping back into space. This process is known as the greenhouse effect.

Here's a simplified explanation of how it works:

1. **Solar radiation**: The sun emits solar radiation, which enters the Earth's atmosphere.
2. **Absorption**: The Earth's surface absorbs some of this radiation, warming the planet.
3. **Infrared radiation**: The Earth's surface emits infrared radiation (heat) back into the atmosphere.
4. **Greenhouse gas absorption**: Greenhouse gases in the atmosphere, such as CO2, CH4, and H2O, absorb some of this infrared radiation.
5. **Re-radiation**: The absorbed radiation is re-radiated in all directions, including back towards the Earth's surface.
6. **Trapping heat**: This re-radiated heat is trapped by the greenhouse gases, pr

2026-03-26 00:32:21,035 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



Question:
How do greenhouse gases affect the Earth's temperature?

Answer:
According to the text, greenhouse gases, including water vapor, carbon dioxide, methane, and nitrous oxide, absorb and emit heat energy in all directions, keeping the Earth's surface and lower atmosphere warm. This process is known as the greenhouse effect.

Here's a step-by-step explanation:

1. **The Sun's energy**: The Sun serves as the primary energy source for the Earth's climate.
2. **Reflection and absorption**: Some of the incoming sunlight is reflected back into space, while the rest is absorbed by the Earth's surface and atmosphere.
3. **Heat emission**: The Earth's surface emits heat (infrared radiation) back into the atmosphere.
4. **Greenhouse gas absorption**: Greenhouse gases in the atmosphere absorb some of this infrared radiation.
5. **Re-radiation**: The absorbed radiation is re-radiated in all directions, including back towards the Earth's surface.
6. **Warming**: The re-radiated heat warms t